<a href="https://colab.research.google.com/github/jaydenchoe/python-lecture-jumptophython-examples/blob/main/2025-1122_gemini_multimedia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [강의 자료] 2025년형 Gemini 2.5 Flash 완전 정복: 나만의 AI 비서 만들기

본 강의 노트는 구글의 최신 **Gemini 2.5 Flash** 모델과 **Google Gen AI SDK (v1.0)**를 사용하여, 초보자도 쉽게 따라 할 수 있는 AI 챗봇 만들기 실습 예제입니다.

### 학습 목표
1.  **최신 환경 설정**: 2025년 표준 SDK (`google-genai`) 설치 및 설정
2.  **멀티모달 기능**: 유튜브 영상 및 오디오 파일을 AI에게 보여주고 대화하기
3.  **도구 사용 (Function Calling)**: AI가 실시간 데이터를 조회하도록 만들기
4.  **JSON 모드**: AI의 답변을 구조화된 데이터로 받기

### 1. 환경 설정 및 라이브러리 설치

가장 먼저, 2025년 11월 기준 최신 구글 AI 도구 상자(`google-genai`)와 유튜브 영상 다운로드 도구(`pytubefix`)를 설치합니다.
* `google-genai`: 구글의 최신 AI를 다루는 필수 라이브러리 (구버전 `google.generativeai` 대체)
* `pytubefix`: 유튜브 영상을 내 컴퓨터로 다운로드하는 라이브러리

In [ ]:
# [실습 1] 필수 라이브러리 설치
!pip install -U google-genai pytubefix

print("설치 완료! 이제 AI를 만날 준비가 되었습니다.")

### 2. API 키 설정 및 첫 인사 (Hello, Gemini!)

설치가 끝났다면 Gemini에게 첫인사를 건네봅니다.
2025년부터는 `genai.Client`를 사용하여 더 직관적으로 AI와 연결합니다.

**[중요]** API 키는 `aistudio.google.com`에서 발급받아야 합니다.

In [ ]:
from google import genai
from google.genai import types
import os

# ---------------------------------------------------------
# [미션] 발급받은 API 키를 아래 따옴표 안에 넣어주세요.
# ---------------------------------------------------------
MY_API_KEY = "여기에_API_키를_붙여넣으세요"

# 클라이언트(AI 비서) 생성
client = genai.Client(api_key=MY_API_KEY)

# Gemini 2.5 Flash 모델에게 인사하기
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="안녕! 너는 2025년의 최신 AI니? 자기소개 좀 해줘."
)

print(response.text)

### 3. [멀티모달] 유튜브 영상 보고 대화하기

Gemini 2.5 Flash는 텍스트뿐만 아니라 **영상(Video)**도 이해할 수 있습니다.
유튜브 링크만 주면 AI가 영상을 보고 내용을 요약해 주는 기능을 만들어봅시다.

**핵심 과정**
1.  `pytubefix`로 영상 다운로드
2.  `client.files.upload`로 구글 클라우드에 업로드
3.  **처리 대기(Processing)**: 영상이 분석될 때까지 기다림 (필수!)
4.  AI에게 질문

In [ ]:
from pytubefix import YouTube
import time

# 1. 유튜브 다운로드 함수
def download_youtube(url, output_path="video.mp4"):
    print(f"📥 다운로드 시작: {url}")
    yt = YouTube(url)
    stream = yt.streams.get_highest_resolution()
    filename = stream.download(filename=output_path)
    print(f"✅ 다운로드 완료: {filename}")
    return filename

# 2. 구글 서버 업로드 함수 (최신 SDK 방식)
def upload_to_gemini(path, mime_type=None):
    print(f"☁️ 업로드 중: {path}")
    file_obj = client.files.upload(
        path=path,
        config={'mime_type': mime_type}
    )
    print(f"✅ 업로드 완료: {file_obj.uri}")
    return file_obj

# 3. 처리 대기 함수 (이게 없으면 에러가 날 수 있어요!)
def wait_for_files_active(files):
    print("⏳ 파일 처리 대기 중...", end="")
    for name in (file.name for file in files):
        file = client.files.get(name=name)
        while file.state.name == "PROCESSING":
            print(".", end="", flush=True)
            time.sleep(5)
            file = client.files.get(name=name)
        if file.state.name != "ACTIVE":
            raise Exception(f"❌ 파일 처리 실패: {file.name}")
    print("\n🚀 준비 완료!")

# --- 실행 코드 ---
# 한국 관광 홍보 영상 (Feel the Rhythm of Korea)
video_url = "https://youtu.be/3P1CnWI62Ik"

# 1. 다운로드
video_path = download_youtube(video_url)

# 2. 업로드
video_file = upload_to_gemini(video_path, mime_type="video/mp4")

# 3. 대기
wait_for_files_active([video_file])

### 4. [퀴즈] 영상 내용을 물어보자!

영상이 준비되었습니다. 이제 Gemini에게 질문을 던져볼 차례입니다.
아래 코드의 빈칸을 채워서 AI에게 영상 분석을 요청해보세요.

**힌트:** `contents` 리스트 안에 우리가 업로드한 `video_file`과 질문(`prompt`)을 같이 넣어야 합니다.

In [ ]:
prompt = "이 영상에 나오는 사람들의 옷차림과 배경을 아주 자세히 묘사해줘."

response = client.models.generate_content(
    model="gemini-2.5-flash",
    # ---------------------------------------------------------
    # [실습 2] 빈칸을 채워보세요! (영상 파일 변수와 질문 변수)
    # ---------------------------------------------------------
    contents=[video_file, prompt]
)

print(response.text)

### 5. [오디오 분석] 음성 파일 듣고 요약하기

이번에는 비디오에서 소리만 추출하여 **오디오 분석**을 해보겠습니다.
회의록 작성이나 긴 강연 요약에 매우 유용합니다.

In [ ]:
# 오디오 전용 다운로드 함수
def download_audio(url, output_path="audio.mp3"):
    yt = YouTube(url)
    stream = yt.streams.get_audio_only()
    filename = stream.download(filename=output_path)
    return filename

# 구글 I/O 키노트 요약해보기 (예시)
audio_url = "https://youtu.be/XEzRZ35urlk"

print("🎧 오디오 처리 시작...")
audio_path = download_audio(audio_url)
audio_file = upload_to_gemini(audio_path, mime_type="audio/mp3")
wait_for_files_active([audio_file])

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[audio_file, "이 오디오의 핵심 내용을 3가지로 요약해줘."]
)

print(response.text)

### 6. [Function Calling] AI에게 '손' 달아주기

AI는 똑똑하지만 오늘 날씨나 현재 재고 같은 실시간 정보는 모릅니다.
**Function Calling**을 사용하면 AI가 필요할 때 우리가 만든 파이썬 코드를 실행하게 할 수 있습니다.

우리는 **스마트폰 판매 챗봇**을 만들어 보겠습니다.
2025년 트렌드인 **`automatic_function_calling=True`** 설정을 사용하면 복잡한 과정 없이 자동으로 함수가 실행됩니다.

In [ ]:
# 1. 가상의 제품 데이터베이스
prod_database = {
    "갤럭시S25": {"재고": 10, "가격": 1500000},
    "아이폰16": {"재고": 0, "가격": 1600000},
}

# 2. 도구(함수) 만들기 - 재고 확인
def is_product_available(product_name: str) -> bool:
    """특정 제품의 재고가 있는지 확인합니다. 재고가 0보다 크면 True입니다."""
    if product_name in prod_database:
        return prod_database[product_name]["재고"] > 0
    return False

# 3. 도구(함수) 만들기 - 가격 확인
def get_product_price(product_name: str) -> int:
    """제품의 가격을 조회합니다. 없는 제품은 0원입니다."""
    if product_name in prod_database:
        return prod_database[product_name]["가격"]
    return 0

# 4. 도구 등록하기
my_tools = [is_product_available, get_product_price]

# 5. AI 챗봇 생성 (자동 도구 사용 설정)
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        tools=my_tools,
        automatic_function_calling=True # ✨ 핵심: AI가 알아서 함수를 사용함
    )
)

print("🤖 판매원 챗봇이 준비되었습니다.")

### 7. [퀴즈] 판매원과 대화하기

이제 챗봇에게 물건을 사러 간 것처럼 질문해보세요.
AI가 스스로 함수를 실행해서 정확한 가격과 재고를 알려줄 겁니다.

In [ ]:
# 질문 예시: "갤럭시S25 가격이 얼마야? 지금 살 수 있어?"
user_question = "갤럭시S25 가격이랑 재고 있는지 알려줘."

response = chat.send_message(user_question)

print(f"사용자: {user_question}")
# ---------------------------------------------------------
# [실습 3] AI의 답변을 출력하는 코드를 작성하세요.
# ---------------------------------------------------------
print(f"AI 판매원: {response.text}")

### 8. [JSON 모드] 퀴즈 생성기 만들기

개발을 하다 보면 AI의 답변을 깔끔한 데이터 형식(JSON)으로 받고 싶을 때가 있습니다.
Gemini 2.5 Flash는 **강제 JSON 출력 모드**를 지원합니다. 이걸 이용해서 영상 내용을 바탕으로 한 '퀴즈 생성기'를 만들어보죠.

In [ ]:
quiz_prompt = """
아까 본 영상 내용을 바탕으로 객관식 퀴즈 1개를 만들어줘.
형식은 반드시 JSON으로 해줘.
예시: {"question": "질문", "options": ["보기1", "보기2", "보기3"], "answer": "정답"}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[video_file, quiz_prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json"  # 무조건 JSON으로 답해!
    )
)

print(response.text)

### 9. 뒷정리 (Cleanup)

구글 클라우드에 올린 파일은 임시 저장소에 있지만, 실습이 끝나면 깔끔하게 지워주는 것이 좋습니다.

In [ ]:
# 클라우드 파일 삭제
try:
    client.files.delete(name=video_file.name)
    client.files.delete(name=audio_file.name)
    print("✅ 클라우드 파일 삭제 완료")
except Exception as e:
    print(f"⚠️ 삭제 중 오류 발생 (이미 삭제됨): {e}")

# 로컬 파일 삭제
if os.path.exists("video.mp4"):
    os.remove("video.mp4")
if os.path.exists("audio.mp3"):
    os.remove("audio.mp3")
print("✨ 모든 실습이 종료되었습니다!")